# Solar deep-learning baselines (Darts)

Training uses data through **2023-09-30 23:00**, split 80/20 into train/validation. Testing begins on **2023-10-01**.



## 1. Environment and data


In [ ]:
!pip install -r requirements/darts.txt


In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, torch, optuna

REPO_ROOT=Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT/"requirements").exists():
    REPO_ROOT=REPO_ROOT.parent
ROOT=REPO_ROOT/"solar"
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))

from sklearn import preprocessing
from pytorch_lightning.callbacks import EarlyStopping
from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.metrics import rmse
from darts.models import TFTModel, TSMixerModel, RNNModel, TiDEModel, DLinearModel, NLinearModel
from darts.utils.likelihood_models import QuantileRegression
from preprocessing import load_data, describe_data
from src.metrics import print_metrics
from power_conversion import convert_solar_result_to_power

data=load_data(ROOT/"data")
print(describe_data(data))


## 2. Configuration


In [ ]:
MODEL="nlinear"  # tft, tsmixer, deepar, tide, dlinear, nlinear
LOAD_EXISTING=True
SAVE_MODEL=True
N_TRIALS=30
BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))
QUANTILES=BASE_QUANTILES
TRAIN_QUANTILES=BASE_QUANTILES  
MODEL_QUANTILES=OUTPUT_QUANTILES  # output/extraction schema only
NUM_SAMPLES=100
SEED=42
TEST_CUTOFF=pd.Timestamp("2023-09-30 23:00")
PAST_COVARIATES=["AIRTEMP","RH_AVG","DEWPT","WS","GHI","DNI"]
FUTURE_PROXY_COLS=[f"{c}_lag24" for c in PAST_COVARIATES]

MODEL_DIR=ROOT/"models"/"darts"; MODEL_DIR.mkdir(parents=True,exist_ok=True)
OUTPUT_DIR=ROOT/"outputs"/"predictions"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
CHECKPOINT_FILES={
    "tft":"tft.darts",
    "tsmixer":"tsmixer.darts",
    "deepar":"deepar.darts",
    "tide":"tide.darts",
    "dlinear":"dlinear.darts",
    "nlinear":"nlinear.darts",
}
MODEL_CLASSES={"tft":TFTModel,"tsmixer":TSMixerModel,"deepar":RNNModel,"tide":TiDEModel,"dlinear":DLinearModel,"nlinear":NLinearModel}
MODEL_PATH=MODEL_DIR/CHECKPOINT_FILES[MODEL]
print("Mode:","LOAD" if LOAD_EXISTING else "TRAIN","|",MODEL_PATH)


## 3. One-year train/validation split and training-only scaling


In [ ]:
df=data.full.sort_values("timestamp").copy()
series=TimeSeries.from_dataframe(df,time_col="timestamp",value_cols="target",freq="h").astype(np.float32)
past_covariates=TimeSeries.from_dataframe(df,time_col="timestamp",value_cols=PAST_COVARIATES,freq="h").astype(np.float32)
future_covariates=TimeSeries.from_dataframe(df,time_col="timestamp",value_cols=FUTURE_PROXY_COLS,freq="h").astype(np.float32)

# Exact one-year section split: pre-test data through 2023-09-30, then 80/20 train/validation.
train,test=series.split_after(TEST_CUTOFF)
train,val=train.split_after(0.8)

past_train,past_test=past_covariates.split_after(TEST_CUTOFF)
past_train,past_val=past_train.split_after(0.8)
future_train,future_test=future_covariates.split_after(TEST_CUTOFF)
future_train,future_val=future_train.split_after(0.8)

# Fit every scaler on training only.
target_scale=Scaler(preprocessing.StandardScaler())
train_t=target_scale.fit_transform(train); val_t=target_scale.transform(val); series_t=target_scale.transform(series)

past_scale=Scaler(preprocessing.StandardScaler())
past_train_t=past_scale.fit_transform(past_train); past_val_t=past_scale.transform(past_val); past_cov_t=past_scale.transform(past_covariates)

future_scale=Scaler(preprocessing.StandardScaler())
future_train_t=future_scale.fit_transform(future_train); future_val_t=future_scale.transform(future_val); future_cov_t=future_scale.transform(future_covariates)

print("train:",train.start_time(),"to",train.end_time(),"rows=",len(train))
print("validation:",val.start_time(),"to",val.end_time(),"rows=",len(val))
print("test:",test.start_time(),"to",test.end_time(),"rows=",len(test))


print("Full target:",series_t.start_time(),"->",series_t.end_time())
print("Past covariates:",past_cov_t.start_time(),"->",past_cov_t.end_time())
print("Future proxy covariates:",future_cov_t.start_time(),"->",future_cov_t.end_time())
assert past_cov_t.end_time()>=series_t.end_time()
assert future_cov_t.end_time()>=series_t.end_time()


## 4. Load existing checkpoint or tune/train


In [ ]:
def early_stopper():
    return EarlyStopping(monitor="val_loss",patience=5,min_delta=1e-3,mode="min",verbose=True)

def build_model(name,p,model_name):
    common=dict(
        output_chunk_length=24,
        batch_size=p["batch_size"],
        n_epochs=100,
        optimizer_kwargs={"lr":p["lr"]},
        random_state=SEED,
        likelihood=QuantileRegression(MODEL_QUANTILES),
        pl_trainer_kwargs={
            "callbacks":[early_stopper()],"accelerator":"auto","devices":1,
            "enable_checkpointing":True,"logger":False,
        },
        model_name=model_name,force_reset=True,save_checkpoints=True,
    )
    if name=="tft":
        return TFTModel(input_chunk_length=p["input_chunk_length"],hidden_size=p["hidden_size"],num_attention_heads=p["num_attention_heads"],lstm_layers=p["lstm_layers"],dropout=p["dropout"],**common)
    if name=="tsmixer":
        return TSMixerModel(input_chunk_length=p["input_chunk_length"],hidden_size=p["hidden_size"],dropout=p["dropout"],**common)
    if name=="deepar":
        return RNNModel(model="LSTM",hidden_dim=p["hidden_size"],input_chunk_length=p["input_chunk_length"],dropout=p["dropout"],training_length=500,**common)
    if name=="tide":
        return TiDEModel(input_chunk_length=p["input_chunk_length"],hidden_size=p["hidden_size"],dropout=p["dropout"],num_encoder_layers=p["num_encoder_layers"],num_decoder_layers=p["num_decoder_layers"],**common)
    if name=="dlinear": return DLinearModel(input_chunk_length=p["input_chunk_length"],**common)
    if name=="nlinear": return NLinearModel(input_chunk_length=p["input_chunk_length"],normalize=False,**common)
    raise ValueError(name)

def fit_model(m, name, verbose=True):
    if name=="deepar":
        return m.fit(train_t,val_series=val_t,future_covariates=future_train_t,val_future_covariates=future_val_t,verbose=verbose)
    return m.fit(
        train_t,val_series=val_t,
        past_covariates=past_train_t,val_past_covariates=past_val_t,
        future_covariates=future_train_t,val_future_covariates=future_val_t,
        verbose=verbose,
    )

def validation_prediction(m,name):
    full_future=future_train_t.append(future_val_t)
    if name=="deepar":
        return m.predict(n=len(val_t),series=train_t,future_covariates=full_future)
    full_past=past_train_t.append(past_val_t)
    return m.predict(n=len(val_t),series=train_t,past_covariates=full_past,future_covariates=full_future)

def tune(name):
    def objective(trial):
        p={
            "input_chunk_length":trial.suggest_categorical("input_chunk_length",[168,336]),
            "batch_size":trial.suggest_categorical("batch_size",[16,32,64]),
            "lr":trial.suggest_float("lr",1e-5,1e-1,log=True),
        }
        if name in {"tft","tsmixer","deepar","tide"}:
            p.update(hidden_size=trial.suggest_categorical("hidden_size",[64,128,256]),dropout=trial.suggest_float("dropout",0.0,0.3))
        if name=="tft":
            p.update(num_attention_heads=trial.suggest_categorical("num_attention_heads",[2,4]),lstm_layers=trial.suggest_categorical("lstm_layers",[1,2]))
        if name=="tide":
            p.update(num_encoder_layers=trial.suggest_int("num_encoder_layers",1,3),num_decoder_layers=trial.suggest_int("num_decoder_layers",1,3))
        torch.manual_seed(SEED)
        m=build_model(name,p,f"{name}_tuning")
        fit_model(m,name,verbose=False)
        pred=validation_prediction(m,name)
        pred=target_scale.inverse_transform(pred); truth=target_scale.inverse_transform(val_t)
        return float(np.mean(rmse(truth,pred)))
    timeout=3600 if name in {"dlinear","nlinear"} else 5400
    study=optuna.create_study(direction="minimize")
    study.optimize(objective,n_trials=N_TRIALS,timeout=timeout,show_progress_bar=True)
    print("Best validation RMSE:",study.best_value); print("Best params:",study.best_params)
    return study.best_params

if LOAD_EXISTING:
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"Checkpoint not found: {MODEL_PATH}. Put the one-year checkpoint there or set LOAD_EXISTING=False.")
    model=MODEL_CLASSES[MODEL].load(str(MODEL_PATH),weights_only=False)
else:
    best_params=tune(MODEL)
    model=build_model(MODEL,best_params,f"{MODEL}_solar_one_year")
    fit_model(model,MODEL,verbose=True)
    if SAVE_MODEL: model.save(str(MODEL_PATH))


## 5. One-year rolling test result


In [ ]:
kwargs=dict(
    series=series_t,
    future_covariates=future_cov_t,
    start=test.start_time(),
    forecast_horizon=24,
    stride=24,
    num_samples=NUM_SAMPLES,
    last_points_only=False,
    retrain=False,
    verbose=True,
)
if MODEL!="deepar":
    kwargs["past_covariates"]=past_cov_t

backtests=model.historical_forecasts(**kwargs)
forecast_scaled=concatenate(backtests)
forecast=target_scale.inverse_transform(forecast_scaled)

gt=series.slice_intersect(forecast); fc=forecast.slice_intersect(gt)
result=pd.DataFrame({"item_id":1,"timestamp":gt.time_index,"target":gt.values().flatten()})
for q in MODEL_QUANTILES:
    result[str(q)]=fc.quantile(q).values().flatten()
result["prediction"]=result["0.5"]


forecast_cols=["prediction"]+[str(q) for q in OUTPUT_QUANTILES if str(q) in result.columns]
result[forecast_cols]=result[forecast_cols].clip(lower=0.0)

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in result.columns:
        result[str(_q)] = np.nan


result["prediction"]=result["0.5"]

result.to_csv(OUTPUT_DIR/f"{MODEL}.csv",index=False)
print_metrics(result,QUANTILES,label=MODEL)

# POA -> solar DC power evaluation.
power_result=convert_solar_result_to_power(result,data.test)

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in power_result.columns:
        power_result[str(_q)] = np.nan
power_result.to_csv(OUTPUT_DIR/f"{MODEL}_power.csv",index=False)
print_metrics(power_result,QUANTILES,label=f"{MODEL} [solar power]")